In [12]:
import numpy as np
import gym
from typing import Optional

In [13]:
class GridWorldEnv(gym.Env):
    
    def __init__(self, size: int = 5, num_pits: int = 3):
        self.size = size  # size of grid 5x5
        self.num_pits = num_pits
        self._agent_location = np.array([-1, -1], dtype=np.int32)
        self._target_location = np.array([-1, -1], dtype=np.int32)
        self._pit_locations = []

        self.observation_space = gym.spaces.Dict({
            "agent": gym.spaces.Box(0, size-1, shape=(2,), dtype=int),
            "target": gym.spaces.Box(0, size-1, shape=(2,), dtype=int),
            "pits": gym.spaces.MultiBinary(size * size)
        })
        self.action_space = gym.spaces.Discrete(4)  # 4 actions: up, down, left, right
        self._action_to_direction = {
            0: np.array([1, 0]),   # right
            1: np.array([0, 1]),   # up
            2: np.array([-1, 0]),  # left
            3: np.array([0, -1]),  # down
        }

    def _get_obs(self):
        pits_flat = np.zeros(self.size * self.size, dtype=np.int8)
        for pit_loc in self._pit_locations:
            idx = pit_loc[0] * self.size + pit_loc[1]
            pits_flat[idx] = 1
        return {
            "agent": self._agent_location,
            "target": self._target_location,
            "pits": pits_flat,
        }
    
    def _get_info(self):
        return {
            "distance": np.linalg.norm(self._agent_location - self._target_location, ord=1),  # Manhattan distance
            "is_in_pit": any(np.array_equal(self._agent_location, pit) for pit in self._pit_locations),
        }
    
    def reset(self, seed: Optional[int] = None, options: Optional[dict] = None):
        super().reset(seed=seed)
        self._agent_location = self.np_random.integers(0, self.size, size=2, dtype=np.int32)
        self._target_location = self.np_random.integers(0, self.size, size=2, dtype=np.int32)
        while(np.array_equal(self._agent_location, self._target_location)):
            self._target_location = self.np_random.integers(0, self.size, size=2, dtype=np.int32)
        
        # Place pits randomly, not on agent or target (and not duplicates)
        pit_locs = set()
        while len(pit_locs) < self.num_pits:
            pit = tuple(self.np_random.integers(0, self.size, size=2, dtype=np.int32))
            if (not np.array_equal(pit, self._agent_location)
                and not np.array_equal(pit, self._target_location)
                and pit not in pit_locs):
                pit_locs.add(pit)
        self._pit_locations = [np.array(pit, dtype=np.int32) for pit in pit_locs]
        return self._get_obs(), self._get_info()

    def step(self, action: int):
        if action not in self._action_to_direction:
            raise ValueError("Invalid action")
        move = self._action_to_direction[action]
        new_pos = self._agent_location + move
        self._agent_location = np.clip(new_pos, 0, self.size - 1)
        
        done = False
        reward = 0

        in_pit = any(np.array_equal(self._agent_location, pit) for pit in self._pit_locations)
        reached_goal = np.array_equal(self._agent_location, self._target_location)
        
        if in_pit:
            done = True
            reward = -1  # Penalty for falling into a pit
        elif reached_goal:
            done = True
            reward = 1   # Reward for reaching target

        info = self._get_info()
        return self._get_obs(), reward, done, False, info

In [14]:
import pygame
import time

# Pygame settings
CELL_SIZE = 100
INFO_HEIGHT = 60
COLORS = {
    'bg': (30, 30, 45),
    'grid': (60, 60, 80),
    'pit': (200, 50, 50),
    'pit_border': (150, 30, 30),
    'goal': (50, 200, 100),
    'goal_border': (30, 150, 70),
    'agent': (80, 150, 255),
    'agent_border': (50, 100, 200),
    'text': (255, 255, 255),
    'text_dim': (150, 150, 170),
    'success': (100, 255, 150),
    'fail': (255, 100, 100)
}

def render_pygame(env, screen, font, step, total_steps, episode, wins, losses):
    """Render the grid world with pygame - enhanced version"""
    size = env.size
    game_height = size * CELL_SIZE
    
    # Background
    screen.fill(COLORS['bg'])
    
    # Draw grid cells with slight gradient effect
    for x in range(size):
        for y in range(size):
            rect = pygame.Rect(x * CELL_SIZE + 2, (size - 1 - y) * CELL_SIZE + 2, CELL_SIZE - 4, CELL_SIZE - 4)
            pygame.draw.rect(screen, COLORS['grid'], rect, border_radius=8)
    
    # Draw pits with glow effect
    for pit in env._pit_locations:
        x, y = pit[0] * CELL_SIZE, (size - 1 - pit[1]) * CELL_SIZE
        # Outer glow
        pygame.draw.rect(screen, COLORS['pit_border'], (x + 5, y + 5, CELL_SIZE - 10, CELL_SIZE - 10), border_radius=12)
        # Inner
        pygame.draw.rect(screen, COLORS['pit'], (x + 10, y + 10, CELL_SIZE - 20, CELL_SIZE - 20), border_radius=8)
        # X mark
        pygame.draw.line(screen, (255, 255, 255), (x + 25, y + 25), (x + CELL_SIZE - 25, y + CELL_SIZE - 25), 4)
        pygame.draw.line(screen, (255, 255, 255), (x + CELL_SIZE - 25, y + 25), (x + 25, y + CELL_SIZE - 25), 4)
    
    # Draw target with pulsing effect
    target = env._target_location
    tx, ty = target[0] * CELL_SIZE, (size - 1 - target[1]) * CELL_SIZE
    pygame.draw.rect(screen, COLORS['goal_border'], (tx + 5, ty + 5, CELL_SIZE - 10, CELL_SIZE - 10), border_radius=12)
    pygame.draw.rect(screen, COLORS['goal'], (tx + 10, ty + 10, CELL_SIZE - 20, CELL_SIZE - 20), border_radius=8)
    # Star/flag symbol
    pygame.draw.polygon(screen, (255, 255, 255), [
        (tx + CELL_SIZE//2, ty + 20),
        (tx + CELL_SIZE//2 + 15, ty + CELL_SIZE - 20),
        (tx + 20, ty + 40),
        (tx + CELL_SIZE - 20, ty + 40),
        (tx + CELL_SIZE//2 - 15, ty + CELL_SIZE - 20)
    ])
    
    # Draw agent with smooth circle
    agent = env._agent_location
    cx = agent[0] * CELL_SIZE + CELL_SIZE // 2
    cy = (size - 1 - agent[1]) * CELL_SIZE + CELL_SIZE // 2
    pygame.draw.circle(screen, COLORS['agent_border'], (cx, cy), CELL_SIZE // 3 + 4)
    pygame.draw.circle(screen, COLORS['agent'], (cx, cy), CELL_SIZE // 3)
    # Eyes
    pygame.draw.circle(screen, (255, 255, 255), (cx - 8, cy - 5), 6)
    pygame.draw.circle(screen, (255, 255, 255), (cx + 8, cy - 5), 6)
    pygame.draw.circle(screen, (0, 0, 0), (cx - 6, cy - 5), 3)
    pygame.draw.circle(screen, (0, 0, 0), (cx + 10, cy - 5), 3)
    
    # Info bar at bottom
    info_rect = pygame.Rect(0, game_height, size * CELL_SIZE, INFO_HEIGHT)
    pygame.draw.rect(screen, (20, 20, 30), info_rect)
    
    # Stats text
    step_text = font.render(f"Step: {step}/{total_steps}", True, COLORS['text'])
    episode_text = font.render(f"Episode: {episode}", True, COLORS['text'])
    wins_text = font.render(f"Wins: {wins}", True, COLORS['success'])
    losses_text = font.render(f"Falls: {losses}", True, COLORS['fail'])
    
    screen.blit(step_text, (15, game_height + 18))
    screen.blit(episode_text, (150, game_height + 18))
    screen.blit(wins_text, (300, game_height + 18))
    screen.blit(losses_text, (400, game_height + 18))
    
    pygame.display.flip()

def show_message(screen, font, message, color, duration=1.0):
    """Display a centered message"""
    size = screen.get_width()
    overlay = pygame.Surface((size, size), pygame.SRCALPHA)
    overlay.fill((0, 0, 0, 150))
    screen.blit(overlay, (0, 0))
    
    text = font.render(message, True, color)
    text_rect = text.get_rect(center=(size // 2, size // 2))
    screen.blit(text, text_rect)
    pygame.display.flip()
    time.sleep(duration)

def auto_play(env, max_steps=50, delay=0.2, seed=None):
    """
    Auto-play the game using a greedy strategy.
    
    Args:
        env: GridWorldEnv instance
        max_steps: Maximum steps to run (stops after this many steps)
        delay: Delay between moves in seconds
        seed: Random seed for reproducibility (None for random)
    """
    pygame.init()
    size = env.size
    screen = pygame.display.set_mode((size * CELL_SIZE, size * CELL_SIZE + INFO_HEIGHT))
    pygame.display.set_caption("🎮 Grid World - Auto Play")
    font = pygame.font.Font(None, 28)
    big_font = pygame.font.Font(None, 48)
    
    # Stats
    episode = 1
    wins = 0
    losses = 0
    
    obs, info = env.reset(seed=seed)
    render_pygame(env, screen, font, 0, max_steps, episode, wins, losses)
    time.sleep(delay)
    
    running = True
    step = 0
    
    while running and step < max_steps:
        # Handle pygame events
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
                break
        
        if not running:
            break
        
        # Greedy strategy: move towards target
        agent = env._agent_location
        target = env._target_location
        dx = target[0] - agent[0]
        dy = target[1] - agent[1]
        
        if abs(dx) >= abs(dy):
            action = 0 if dx > 0 else 2
        else:
            action = 1 if dy > 0 else 3
        
        obs, reward, done, _, info = env.step(action)
        step += 1
        render_pygame(env, screen, font, step, max_steps, episode, wins, losses)
        time.sleep(delay)
        
        if done:
            if reward > 0:
                wins += 1
                show_message(screen, big_font, f"🎉 GOAL! ({step} steps)", COLORS['success'], 0.8)
            else:
                losses += 1
                show_message(screen, big_font, "💀 PIT!", COLORS['fail'], 0.8)
            
            # Start new episode
            episode += 1
            obs, info = env.reset()
            render_pygame(env, screen, font, step, max_steps, episode, wins, losses)
            time.sleep(delay)
    
    # Final summary
    show_message(screen, big_font, f"Done! Wins: {wins} Falls: {losses}", COLORS['text'], 2)
    pygame.quit()
    print(f"\n=== Game Summary ===")
    print(f"Total Steps: {step}")
    print(f"Episodes: {episode}")
    print(f"Wins: {wins}, Falls: {losses}")
    print(f"Win Rate: {wins/(wins+losses)*100:.1f}%" if (wins+losses) > 0 else "N/A")




In [17]:
# ========== RUN THE GAME ==========
# Adjust these parameters as you like:
env = GridWorldEnv(size=5, num_pits=3)
auto_play(
    env, 
    max_steps=50,    # Change this to control how many steps
    delay=0.3,       # Speed of animation (lower = faster)
    seed=42          # Set to None for random games
)


=== Game Summary ===
Total Steps: 50
Episodes: 19
Wins: 13, Falls: 5
Win Rate: 72.2%
